In [1]:
# Cell 1 - imports, paths, backup helper

import shutil
from pathlib import Path
from datetime import datetime

import pandas as pd

# Base paths
DATA_DIR   = Path(r"C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data")
OUTPUT_DIR = DATA_DIR / "output"

# Raw CSVs (source data)
RAW_FILES = {
    "battery":      DATA_DIR / "battery.csv",
    "body":         DATA_DIR / "body.csv",
    "engine":       DATA_DIR / "engine.csv",
    "transmission": DATA_DIR / "transmission.csv",
    "tyre":         DATA_DIR / "tyre.csv",
}

# Inference CSVs
INF_FILES = {
    "battery":      OUTPUT_DIR / "battery_inference.csv",
    "body":         OUTPUT_DIR / "body_inference.csv",
    "engine":       OUTPUT_DIR / "engine_inference.csv",
    "transmission": OUTPUT_DIR / "transmission_inference.csv",
    "tyre":         OUTPUT_DIR / "tyre_inference.csv",
}

FEATURES_JSON_PATH = DATA_DIR / "features.json"

def timestamp_suffix():
    """Return a filesystem-safe timestamp for backup filenames."""
    return datetime.now().strftime("%Y%m%dT%H%M%S")

def backup_file(path: Path):
    """
    Create a backup of 'path' as 'path.bak.<timestamp>'.
    Returns the backup path.
    """
    if not path.exists():
        raise FileNotFoundError(f"Cannot backup missing file: {path}")
    suffix = timestamp_suffix()
    backup_path = Path(str(path) + f".bak.{suffix}")
    shutil.copy2(path, backup_path)
    print(f"Backup created: {backup_path}")
    return backup_path


In [2]:
# Cell 2 - load canonical timestamp column from engine_inference.csv

engine_inf_path = INF_FILES["engine"]
if not engine_inf_path.exists():
    raise FileNotFoundError(f"Engine inference CSV not found at: {engine_inf_path}")

# Read only the timestamp column to avoid wasting memory
engine_inf_df = pd.read_csv(
    engine_inf_path,
    usecols=["timestamp"],
    dtype=str,
    keep_default_na=False,
    na_values=[""],
    engine="python",
)

canonical_ts = engine_inf_df["timestamp"].astype(str)
n_rows = len(canonical_ts)

print(f"Loaded {n_rows} timestamps from {engine_inf_path}")
print("First 5 timestamps:", canonical_ts.head().tolist())


Loaded 31799 timestamps from C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\engine_inference.csv
First 5 timestamps: ['2024-07-05T07:22:10+0000', '2024-07-05T07:22:11+0000', '2024-07-05T07:22:12+0000', '2024-07-05T07:22:13+0000', '2024-07-05T07:22:14+0000']


In [4]:
# Cell 3 - helper to overwrite timestamp column in a CSV with canonical engine timestamps (FIXED)

def overwrite_timestamp_column(csv_path: Path, ts_series: pd.Series, timestamp_col: str = "timestamp"):
    """
    Overwrite the 'timestamp' column of csv_path with values from ts_series.
    - Makes a backup first.
    - Ensures row counts match.
    - Writes CSV back in place.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Target CSV not found: {csv_path}")

    # Backup first
    backup_file(csv_path)

    # Read entire CSV as strings to avoid type surprises
    df = pd.read_csv(
        csv_path,
        dtype=str,
        keep_default_na=False,
        na_values=[""],
        engine="python",
    )

    if timestamp_col not in df.columns:
        raise KeyError(f"File {csv_path} does not have a '{timestamp_col}' column.")

    if len(df) != len(ts_series):
        raise ValueError(
            f"Row count mismatch for {csv_path}: "
            f"file has {len(df)} rows, engine_inference has {len(ts_series)} rows."
        )

    # Overwrite timestamp column
    df[timestamp_col] = ts_series.values

    # Write back in place
    df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",   # <-- fixed name
    )
    print(f"Updated timestamp column in: {csv_path}")


# --- Apply to all inference CSVs ---
print("Updating inference CSVs with canonical timestamps from engine_inference.csv ...")
for module, path in INF_FILES.items():
    overwrite_timestamp_column(path, canonical_ts, timestamp_col="timestamp")

# --- Apply to all raw CSVs ---
print("\nUpdating raw CSVs with canonical timestamps from engine_inference.csv ...")
for module, path in RAW_FILES.items():
    overwrite_timestamp_column(path, canonical_ts, timestamp_col="timestamp")

print("\nAll CSV timestamp columns updated successfully.")


Updating inference CSVs with canonical timestamps from engine_inference.csv ...
Backup created: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\battery_inference.csv.bak.20251116T171601
Updated timestamp column in: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\battery_inference.csv
Backup created: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\body_inference.csv.bak.20251116T171605
Updated timestamp column in: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\body_inference.csv
Backup created: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\engine_inference.csv.bak.20251116T171608
Updated timestamp column in: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\engine_inference.csv
Backup created: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\transmission_inference.csv.bak.20251116T171612
Updated timestamp column in: C:\Us

In [5]:
# Cell X – Update features.json after timestamp changes

import json
from pathlib import Path
import pandas as pd
from datetime import datetime
import shutil

# Paths
DATA_DIR = Path(r"C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data")
FEATURES_JSON_PATH = DATA_DIR / "features.json"

RAW_FILES = {
    "battery.csv":      DATA_DIR / "battery.csv",
    "body.csv":         DATA_DIR / "body.csv",
    "engine.csv":       DATA_DIR / "engine.csv",
    "transmission.csv": DATA_DIR / "transmission.csv",
    "tyre.csv":         DATA_DIR / "tyre.csv",
}

def backup_file(path: Path):
    """Backup a file as file.bak.<timestamp>"""
    ts = datetime.now().strftime("%Y%m%dT%H%M%S")
    backup_path = Path(str(path) + f".bak.{ts}")
    shutil.copy2(path, backup_path)
    print(f"Backup created: {backup_path}")
    return backup_path


# -----------------------------
# 1. Backup features.json
# -----------------------------
if not FEATURES_JSON_PATH.exists():
    raise FileNotFoundError(f"features.json not found: {FEATURES_JSON_PATH}")

backup_file(FEATURES_JSON_PATH)


# -----------------------------
# 2. Load features.json
# -----------------------------
with FEATURES_JSON_PATH.open("r", encoding="utf-8") as f:
    feat = json.load(f)


# -----------------------------
# 3. Update generated_at
# -----------------------------
feat["generated_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%fZ")


# -----------------------------
# 4. Update timestamp metadata for each raw CSV
# -----------------------------

def update_timestamp_metadata(file_name: str, csv_path: Path, features_dict: dict, sample_n: int = 5):
    """
    Update timestamp column metadata inside features.json for one CSV.
    - refresh unique_count
    - refresh sample_values
    """
    if not csv_path.exists():
        print(f"WARNING: Raw file missing on disk: {csv_path}")
        return

    # Load timestamp column
    df = pd.read_csv(
        csv_path,
        usecols=["timestamp"],
        dtype=str,
        keep_default_na=False,
        na_values=[""],
        engine="python"
    )
    ts = df["timestamp"].astype(str)

    unique_count = int(ts.nunique())
    sample_values = ts.head(sample_n).tolist()

    # Navigate inside features.json
    file_block = features_dict["files"].get(file_name, None)
    if not file_block:
        print(f"WARNING: {file_name} missing in features.json structure.")
        return

    col_entries = file_block.get("columns", [])
    ts_entry = next((c for c in col_entries if c.get("name") == "timestamp"), None)

    if not ts_entry:
        print(f"WARNING: No timestamp entry in features.json for {file_name}")
        return

    # Update metadata
    ts_entry["unique_count"] = unique_count
    ts_entry["sample_values"] = sample_values

    print(f"{file_name}: updated timestamp metadata:")
    print(f"  unique_count = {unique_count}")
    print(f"  sample_values = {sample_values}")


# Update for all 5 raw files
for file_name, csv_path in RAW_FILES.items():
    update_timestamp_metadata(file_name, csv_path, feat)


# -----------------------------
# 5. Save updated features.json in place
# -----------------------------
with FEATURES_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(feat, f, indent=2, ensure_ascii=False)

print("\nfeatures.json updated successfully.")


Backup created: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\features.json.bak.20251116T171802


C:\Users\ishaa\AppData\Local\Temp\ipykernel_11040\3210822802.py:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  feat["generated_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%fZ")


battery.csv: updated timestamp metadata:
  unique_count = 31799
  sample_values = ['2024-07-05T07:22:10+0000', '2024-07-05T07:22:11+0000', '2024-07-05T07:22:12+0000', '2024-07-05T07:22:13+0000', '2024-07-05T07:22:14+0000']
body.csv: updated timestamp metadata:
  unique_count = 31799
  sample_values = ['2024-07-05T07:22:10+0000', '2024-07-05T07:22:11+0000', '2024-07-05T07:22:12+0000', '2024-07-05T07:22:13+0000', '2024-07-05T07:22:14+0000']
engine.csv: updated timestamp metadata:
  unique_count = 31799
  sample_values = ['2024-07-05T07:22:10+0000', '2024-07-05T07:22:11+0000', '2024-07-05T07:22:12+0000', '2024-07-05T07:22:13+0000', '2024-07-05T07:22:14+0000']
transmission.csv: updated timestamp metadata:
  unique_count = 31799
  sample_values = ['2024-07-05T07:22:10+0000', '2024-07-05T07:22:11+0000', '2024-07-05T07:22:12+0000', '2024-07-05T07:22:13+0000', '2024-07-05T07:22:14+0000']
tyre.csv: updated timestamp metadata:
  unique_count = 31799
  sample_values = ['2024-07-05T07:22:10+0000',

In [6]:
# Cell – Verify timestamps in raw & inference CSVs and features.json

import json
from pathlib import Path
import pandas as pd

# -----------------------------
# Paths & config
# -----------------------------
DATA_DIR   = Path(r"C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data")
OUTPUT_DIR = DATA_DIR / "output"

RAW_FILES = {
    "battery":      DATA_DIR / "battery.csv",
    "body":         DATA_DIR / "body.csv",
    "engine":       DATA_DIR / "engine.csv",
    "transmission": DATA_DIR / "transmission.csv",
    "tyre":         DATA_DIR / "tyre.csv",
}

INF_FILES = {
    "battery":      OUTPUT_DIR / "battery_inference.csv",
    "body":         OUTPUT_DIR / "body_inference.csv",
    "engine":       OUTPUT_DIR / "engine_inference.csv",
    "transmission": OUTPUT_DIR / "transmission_inference.csv",
    "tyre":         OUTPUT_DIR / "tyre_inference.csv",
}

FEATURES_JSON_PATH = DATA_DIR / "features.json"

SAMPLE_N = 5  # how many timestamps we expect to see in sample_values


# -----------------------------
# 1. Load canonical timestamps from engine_inference.csv
# -----------------------------
engine_inf_path = INF_FILES["engine"]
if not engine_inf_path.exists():
    raise FileNotFoundError(f"Engine inference CSV not found at: {engine_inf_path}")

engine_inf_df = pd.read_csv(
    engine_inf_path,
    usecols=["timestamp"],
    dtype=str,
    keep_default_na=False,
    na_values=[""],
    engine="python"
)
canonical_ts = engine_inf_df["timestamp"].astype(str).reset_index(drop=True)
n_rows = len(canonical_ts)

print(f"Canonical timestamps loaded from: {engine_inf_path}")
print(f"  Rows: {n_rows}")
print(f"  First {min(SAMPLE_N, n_rows)} timestamps:", canonical_ts.head(SAMPLE_N).tolist())
print()


# -----------------------------
# 2. Verify raw CSVs match canonical timestamps
# -----------------------------
raw_ok = True
for module, csv_path in RAW_FILES.items():
    if not csv_path.exists():
        print(f"[RAW][{module}] MISSING file: {csv_path}")
        raw_ok = False
        continue

    df = pd.read_csv(
        csv_path,
        usecols=["timestamp"],
        dtype=str,
        keep_default_na=False,
        na_values=[""],
        engine="python"
    )
    ts = df["timestamp"].astype(str).reset_index(drop=True)

    if len(ts) != n_rows:
        print(f"[RAW][{module}] Row count mismatch: file={len(ts)}, canonical={n_rows}")
        raw_ok = False
        continue

    equal = ts.equals(canonical_ts)
    if not equal:
        # Find first mismatch index for debug
        mismatch_idx = (ts != canonical_ts).idxmax()
        print(f"[RAW][{module}] TIMESTAMP MISMATCH at index {mismatch_idx}: "
              f"file_ts={ts.iloc[mismatch_idx]!r}, canonical_ts={canonical_ts.iloc[mismatch_idx]!r}")
        raw_ok = False
    else:
        print(f"[RAW][{module}] OK – timestamps match canonical exactly.")

print()


# -----------------------------
# 3. Verify inference CSVs match canonical timestamps
# -----------------------------
inf_ok = True
for module, csv_path in INF_FILES.items():
    if not csv_path.exists():
        print(f"[INF][{module}] MISSING file: {csv_path}")
        inf_ok = False
        continue

    df = pd.read_csv(
        csv_path,
        usecols=["timestamp"],
        dtype=str,
        keep_default_na=False,
        na_values=[""],
        engine="python"
    )
    ts = df["timestamp"].astype(str).reset_index(drop=True)

    if len(ts) != n_rows:
        print(f"[INF][{module}] Row count mismatch: file={len(ts)}, canonical={n_rows}")
        inf_ok = False
        continue

    equal = ts.equals(canonical_ts)
    if not equal:
        mismatch_idx = (ts != canonical_ts).idxmax()
        print(f"[INF][{module}] TIMESTAMP MISMATCH at index {mismatch_idx}: "
              f"file_ts={ts.iloc[mismatch_idx]!r}, canonical_ts={canonical_ts.iloc[mismatch_idx]!r}")
        inf_ok = False
    else:
        print(f"[INF][{module}] OK – timestamps match canonical exactly.")

print()


# -----------------------------
# 4. Verify features.json matches raw CSV timestamps
# -----------------------------
feat_ok = True

if not FEATURES_JSON_PATH.exists():
    raise FileNotFoundError(f"features.json not found at: {FEATURES_JSON_PATH}")

with FEATURES_JSON_PATH.open("r", encoding="utf-8") as f:
    feat = json.load(f)

files_block = feat.get("files", {})

for module, csv_path in RAW_FILES.items():
    file_name = csv_path.name  # e.g. "battery.csv"
    if file_name not in files_block:
        print(f"[FEATURES][{module}] {file_name} missing in features.json 'files' block.")
        feat_ok = False
        continue

    # Load timestamp column from raw CSV again for comparison
    df = pd.read_csv(
        csv_path,
        usecols=["timestamp"],
        dtype=str,
        keep_default_na=False,
        na_values=[""],
        engine="python"
    )
    ts = df["timestamp"].astype(str).reset_index(drop=True)
    unique_count = int(ts.nunique())
    expected_samples = ts.head(SAMPLE_N).tolist()

    file_block = files_block[file_name]
    cols_meta = file_block.get("columns", [])
    ts_meta = next((c for c in cols_meta if c.get("name") == "timestamp"), None)

    if ts_meta is None:
        print(f"[FEATURES][{module}] No 'timestamp' column metadata for {file_name}")
        feat_ok = False
        continue

    meta_unique = ts_meta.get("unique_count")
    meta_samples = ts_meta.get("sample_values", [])

    # Compare unique_count
    if meta_unique != unique_count:
        print(f"[FEATURES][{module}] unique_count mismatch for {file_name}: "
              f"features.json={meta_unique}, actual={unique_count}")
        feat_ok = False
    else:
        print(f"[FEATURES][{module}] unique_count OK ({unique_count}).")

    # Compare sample_values (allow features.json to have <= SAMPLE_N samples)
    k = min(len(meta_samples), SAMPLE_N)
    if meta_samples[:k] != expected_samples[:k]:
        print(f"[FEATURES][{module}] sample_values mismatch for {file_name}:")
        print(f"  features.json={meta_samples}")
        print(f"  actual       ={expected_samples}")
        feat_ok = False
    else:
        print(f"[FEATURES][{module}] sample_values OK (first {k} entries).")

print()


# -----------------------------
# 5. Final summary
# -----------------------------
print("=== SUMMARY ===")
print(f"Raw CSV timestamps OK?        {'YES' if raw_ok else 'NO'}")
print(f"Inference CSV timestamps OK?  {'YES' if inf_ok else 'NO'}")
print(f"features.json metadata OK?    {'YES' if feat_ok else 'NO'}")

if raw_ok and inf_ok and feat_ok:
    print("\nAll checks PASSED: timestamps and features.json are consistent.")
else:
    print("\nOne or more checks FAILED. See messages above for details.")


Canonical timestamps loaded from: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\engine_inference.csv
  Rows: 31799
  First 5 timestamps: ['2024-07-05T07:22:10+0000', '2024-07-05T07:22:11+0000', '2024-07-05T07:22:12+0000', '2024-07-05T07:22:13+0000', '2024-07-05T07:22:14+0000']

[RAW][battery] OK – timestamps match canonical exactly.
[RAW][body] OK – timestamps match canonical exactly.
[RAW][engine] OK – timestamps match canonical exactly.
[RAW][transmission] OK – timestamps match canonical exactly.
[RAW][tyre] OK – timestamps match canonical exactly.

[INF][battery] OK – timestamps match canonical exactly.
[INF][body] OK – timestamps match canonical exactly.
[INF][engine] OK – timestamps match canonical exactly.
[INF][transmission] OK – timestamps match canonical exactly.
[INF][tyre] OK – timestamps match canonical exactly.

[FEATURES][battery] unique_count OK (31799).
[FEATURES][battery] sample_values OK (first 5 entries).
[FEATURES][body] unique_count OK (31